# Simulated Microscope – Full FOV Stimulation Pipeline

This notebook is identical to `01_full_FOV_stimulation_ERK_new_API.ipynb` except the
microscope is replaced by a `SimulatedMicroscope` backed by
[microscope-cell-simulation](https://github.com/dario-bassi/microscope-cell-simulation).
No physical hardware is required.

```bash
pip ii
```

### Import required libraries

In [1]:
import os
import time
from rtm_pymmcore.data_structures import Channel, StimTreatment, SegmentationMethod
import rtm_pymmcore.utils as utils
from pprint import pprint
import pandas as pd

### Experimental Settings

In [ ]:
# ---------------------------------------------------------------------------
# Only this cell differs from the real-hardware notebook.
# Swap SimulatedMicroscope ↔ Jungfrau (or any other microscope) as needed.
# ---------------------------------------------------------------------------
from rtm_pymmcore.microscope.SimulatedMicroscope import SimulatedMicroscope

mic = SimulatedMicroscope(
    nb_cells=120,
    cell_type="optogenetic",
    viewport_width=512,
    viewport_height=512,
    rng_seed=42,
    brownian_d=5,
    channel_rendering_modes={0: 1, 1: 2},
)
mic.mmc.setChannelGroup("Channel")

In [3]:
## Configuration options - set experiment timing, storage and stimulation parameters
# General timing and frame counts:
FIRST_FRAME_STIMULATION = 5
N_FRAMES_PHASE_1 = 20  # number of timesteps in phase 1 (pre-drug)

SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

TIME_BETWEEN_TIMESTEPS = 5  # seconds between timesteps
TIME_PER_FOV = 3.75  # seconds per FOV (camera + stage moves + overhead)

ADD_STIM_EXPOSURE_GROUP = False
REGULAR_SPACING_BETWEEN_STIMULATIONS = False

## Storage path for the experiment
base_path = os.path.join(os.getcwd(), "simulation_output")
experiment_name = "sim_full_FOV_stim"
path = os.path.join(base_path, experiment_name)

## Channels
channels = []
channels.append(Channel(name="Nucleus", exposure=300))
channels.append(Channel(name="Membrane", exposure=200))


# Conditions
condition = ["Drug"]
n_fovs_per_well = None

# Stimulation parameters
stim_phase_1 = [
    StimTreatment(
        treatment_name="Priming Phase 1 pre Drug",
        stim_timestep=(tuple(range(5, 20, 1))),
        stim_exposure_list=(100,) * 15,
        stim_power=10,
        stim_channel_name="Stim",
        stim_channel_group="Channel",
        stim_channel_device_name="Spectra",
        stim_channel_power_property_name="Cyan_Level",
    )
]

for stim_phase in [stim_phase_1]:
    utils.print_stim_exposures_timesteps(stim_phase)

## Define the Tools that you are using for the experiment
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.optocheck_fe import OptoCheckFE
from rtm_pymmcore.segmentation.base_segmentation import SegmentatorBinary

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=SegmentatorBinary(),
        use_channel=0,
        save_tracked=True,
    )
]

stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy(search_range=50)
optocheck = OptoCheckFE(used_mask="labels")

from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=None,
)
mic.set_pipeline(pipeline=pipeline)

Pattern Name:  Priming Phase 1 pre Drug
100 at 5
100 at 6
100 at 7
100 at 8
100 at 9
100 at 10
100 at 11
100 at 12
100 at 13
100 at 14
100 at 15
100 at 16
100 at 17
100 at 18
100 at 19

Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore.worktrees\dev-add_simulation_mic\simulation_output\sim_full_FOV_stim\raw already exists
Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore.worktrees\dev-add_simulation_mic\simulation_output\sim_full_FOV_stim\tracks already exists
Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore.worktrees\dev-add_simulation_mic\simulation_output\sim_full_FOV_stim\stim_mask already exists
Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore.worktrees\dev-add_simulation_mic\simulation_output\sim_full_FOV_stim\stim already exists
Directory c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore.worktrees\dev-add_simulation_mic\simulation_output\sim_full_FOV_stim\particles already exists
Directory c:\Users\Alex\Programmieren\01_git\PhD

### GUI - Napari Micromanager

#### Load GUI

In [4]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

### Map Experiment to FOVs

Use the GUI to set FOV positions (same workflow as with a real microscope),
**or** create them programmatically below.

### Use FOVs to generate dataframe for acquisition

In [5]:
fovs = utils.generate_fov_objects(mic, viewer=viewer)


df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_1,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    phase_name="PreDrug",
    phase_id=0,
    condition=condition,
)
df_acquire = utils.apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_phase_1,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire

Total Experiment Time: 0.02638888888888889h
Doing 1 experiment per stim condition


c:\Users\Alex\Programmieren\01_git\PhD\rtm-pymmcore.worktrees\dev-add_simulation_mic\rtm_pymmcore\utils.py:178: FutureWarning: The `_dock_widgets` property is private and should not be used in any plugin code. Please use the `dock_widgets` property instead.
  data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions


,fov_object,fov,fov_x,fov_y,fov_z,fov_name,timestep,time,channels,fname,...,treatment_name,stim_timestep,stim_exposure_list,stim_power,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,494.0,494.0,0.0,0,0,0.0,"({'name': 'Nucleus', 'exposure': 300, 'group':...",000_00_00000,...,Priming Phase 1 pre Drug,"(5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Stim,Channel,Spectra,Cyan_Level,0.0,False
1,<rtm_pymmcore.data_structures.Fov object at 0x...,0,494.0,494.0,0.0,0,1,5.0,"({'name': 'Nucleus', 'exposure': 300, 'group':...",000_00_00001,...,Priming Phase 1 pre Drug,"(5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Stim,Channel,Spectra,Cyan_Level,0.0,False
2,<rtm_pymmcore.data_structures.Fov object at 0x...,0,494.0,494.0,0.0,0,2,10.0,"({'name': 'Nucleus', 'exposure': 300, 'group':...",000_00_00002,...,Priming Phase 1 pre Drug,"(5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Stim,Channel,Spectra,Cyan_Level,0.0,False
3,<rtm_pymmcore.data_structures.Fov object at 0x...,0,494.0,494.0,0.0,0,3,15.0,"({'name': 'Nucleus', 'exposure': 300, 'group':...",000_00_00003,...,Priming Phase 1 pre Drug,"(5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Stim,Channel,Spectra,Cyan_Level,0.0,False
4,<rtm_pymmcore.data_structures.Fov object at 0x...,0,494.0,494.0,0.0,0,4,20.0,"({'name': 'Nucleus', 'exposure': 300, 'group':...",000_00_00004,...,Priming Phase 1 pre Drug,"(5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Stim,Channel,Spectra,Cyan_Level,0.0,False
5,<rtm_pymmcore.data_structures.Fov object at 0x...,0,494.0,494.0,0.0,0,5,25.0,"({'name': 'Nucleus', 'exposure': 300, 'group':...",000_00_00005,...,Priming Phase 1 pre Drug,"(5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Stim,Channel,Spectra,Cyan_Level,100.0,True
6,<rtm_pymmcore.data_structures.Fov object at 0x...,0,494.0,494.0,0.0,0,6,30.0,"({'name': 'Nucleus', 'exposure': 300, 'group':...",000_00_00006,...,Priming Phase 1 pre Drug,"(5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Stim,Channel,Spectra,Cyan_Level,100.0,True
7,<rtm_pymmcore.data_structures.Fov object at 0x...,0,494.0,494.0,0.0,0,7,35.0,"({'name': 'Nucleus', 'exposure': 300, 'group':...",000_00_00007,...,Priming Phase 1 pre Drug,"(5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Stim,Channel,Spectra,Cyan_Level,100.0,True
8,<rtm_pymmcore.data_structures.Fov object at 0x...,0,494.0,494.0,0.0,0,8,40.0,"({'name': 'Nucleus', 'exposure': 300, 'group':...",000_00_00008,...,Priming Phase 1 pre Drug,"(5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Stim,Channel,Spectra,Cyan_Level,100.0,True
9,<rtm_pymmcore.data_structures.Fov object at 0x...,0,494.0,494.0,0.0,0,9,45.0,"({'name': 'Nucleus', 'exposure': 300, 'group':...",000_00_00009,...,Priming Phase 1 pre Drug,"(5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17...","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Stim,Channel,Spectra,Cyan_Level,100.0,True


### Run experiment

In [ ]:
for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()

except:
    pass

mic.run_experiment(df_acquire)

In [7]:
mic.post_experiment()
time.sleep(10)

utils.generate_exp_data_from_tracks(path)

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Function to re-connect link with GUI if manually broken

The following functions can be used to manually re-make the connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [ ]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()